# MF 731 Homework 5 - Scenario analysis of a delta-hedged spread-squared option

Code only. Write-up and discussion are in `hw5_written.ipynb`.

Long one unit of the option paying $(S_T^{(1)} - S_T^{(2)})^2$, delta-hedged in
both stocks with the hedge held fixed for five days. The closed-form price in
(a) is derived by hand (attached); this notebook evaluates it over the 3888
scenarios of the sheet and outputs the risk measures of (b) and (c).

In [1]:
import itertools
import numpy as np
from fractions import Fraction

np.set_printoptions(suppress=True)

### Parameters (the sheet, at $t = 0$)

In [2]:
r = 0.0132
sigma1_0, sigma2_0, rho_0 = 0.2214, 0.3036, 0.237
T = 0.25
S1_0, S2_0 = 158.12, 170.33
Delta = 1 / 252
h = 5 * Delta                       # 5-day horizon, the revaluation date

# mu1 = 0.15475 and mu2 = 0.18312 are given on the sheet but play no part:
# the price is risk neutral and the scenarios fix the log returns directly.

### (a) Closed-form price and the two deltas

Derived by hand (attached). $V$ is homogeneous of degree 2 in $(s_1, s_2)$, so $s_1 h^{(1)} + s_2 h^{(2)} = 2V$.

In [3]:
def V(t, s1, s2, sig1, sig2, rho):
    """Price at time t of the payoff (S_T^(1) - S_T^(2))^2."""
    tau = T - t
    return np.exp(r * tau) * (s1**2 * np.exp(sig1**2 * tau)
                              - 2 * s1 * s2 * np.exp(sig1 * sig2 * rho * tau)
                              + s2**2 * np.exp(sig2**2 * tau))


def deltas(t, s1, s2, sig1, sig2, rho):
    """h^(1) = dV/ds1, h^(2) = dV/ds2."""
    tau = T - t
    h1 = 2 * np.exp(r * tau) * (s1 * np.exp(sig1**2 * tau) - s2 * np.exp(sig1 * sig2 * rho * tau))
    h2 = 2 * np.exp(r * tau) * (s2 * np.exp(sig2**2 * tau) - s1 * np.exp(sig1 * sig2 * rho * tau))
    return h1, h2


# finite-difference check of the deltas
eps = 1e-5
fd1 = (V(0, S1_0 + eps, S2_0, sigma1_0, sigma2_0, rho_0) - V(0, S1_0 - eps, S2_0, sigma1_0, sigma2_0, rho_0)) / (2 * eps)
fd2 = (V(0, S1_0, S2_0 + eps, sigma1_0, sigma2_0, rho_0) - V(0, S1_0, S2_0 - eps, sigma1_0, sigma2_0, rho_0)) / (2 * eps)
print('finite-difference deltas: %.4f  %.4f' % (fd1, fd2))

finite-difference deltas: -21.9525  31.2019


### Portfolio at $t = 0$

In [4]:
h1, h2 = deltas(0, S1_0, S2_0, sigma1_0, sigma2_0, rho_0)
V_opt0 = V(0, S1_0, S2_0, sigma1_0, sigma2_0, rho_0)
V_0 = V_opt0 - h1 * S1_0 - h2 * S2_0

print('h1 = %.4f   h2 = %.4f' % (h1, h2))
print('option value V(0) = %.4f   portfolio V_0 = %.4f' % (V_opt0, V_0))
print('h1*S1 + h2*S2 = %.4f  (= 2 V(0) by homogeneity)' % (h1 * S1_0 + h2 * S2_0))

# where the option value comes from: intrinsic part vs the variance part
intrinsic = np.exp(r * T) * (S1_0 - S2_0)**2
print('e^{rT}(S1-S2)^2 = %.2f   rest = %.2f' % (intrinsic, V_opt0 - intrinsic))
spread_var = sigma1_0**2 * S1_0**2 - 2 * rho_0 * sigma1_0 * sigma2_0 * S1_0 * S2_0 + sigma2_0**2 * S2_0**2
print('first-order approximation of the rest, e^{rT} T (s1^2 sig1^2 - 2 rho sig1 sig2 s1 s2 + s2^2 sig2^2) = %.2f'
      % (np.exp(r * T) * T * spread_var))

h1 = -21.9525   h2 = 31.2019
option value V(0) = 921.7399   portfolio V_0 = -921.7399
h1*S1 + h2*S2 = 1843.4797  (= 2 V(0) by homogeneity)
e^{rT}(S1-S2)^2 = 149.58   rest = 772.16
first-order approximation of the rest, e^{rT} T (s1^2 sig1^2 - 2 rho sig1 sig2 s1 s2 + s2^2 sig2^2) = 762.91


### Scenario grid and weights (sheet, (b) and (c))

In [5]:
X_grid = [-0.20, -0.10, -0.05, 0.05, 0.10, 0.20]       # X^(i)_{t+5D}, i = 1, 2
K_grid = [0.5, 0.75, 1.25, 1.5, 1.75, 2.0]              # sigma^(i)_{t+5D} / sigma^(i)_t
R_grid = [-0.5, 0.0, 0.5]                               # rho_{t+5D} - rho_t

wX = {0.20: 0.5, 0.10: 0.75, 0.05: 1.0}                                 # keyed by |X|
wK = {0.5: 0.5, 2.0: 0.5, 0.75: 1.25, 1.75: 1.25, 1.25: 0.75, 1.5: 0.75}
wR = {0.0: 0.5, 0.5: 1.25}                                              # keyed by |d rho|


def weight(x1, x2, k1, k2, dr):
    return wX[abs(x1)] * wX[abs(x2)] * wK[k1] * wK[k2] * wR[abs(dr)]


# the sheet's example: X1 = -0.1, X2 = 0.2, k1 = 2, k2 = 0.5, d rho = -0.5 has weight 15/128
w_ex = weight(-0.1, 0.2, 2.0, 0.5, -0.5)
print('example weight = %.6f = %s' % (w_ex, Fraction(w_ex).limit_denominator(1000)))

example weight = 0.117188 = 15/128


### Loop over all $N = 3888$ scenarios, hedge held fixed

In [6]:
results = []
for x1, x2, k1, k2, dr in itertools.product(X_grid, X_grid, K_grid, K_grid, R_grid):
    S1 = S1_0 * np.exp(x1)
    S2 = S2_0 * np.exp(x2)
    sig1, sig2, rho = k1 * sigma1_0, k2 * sigma2_0, rho_0 + dr
    V_h = V(h, S1, S2, sig1, sig2, rho) - h1 * S1 - h2 * S2     # same h1, h2 as at t = 0
    loss = -(V_h - V_0)
    w = weight(x1, x2, k1, k2, dr)
    results.append(dict(X1=x1, X2=x2, k1=k1, k2=k2, drho=dr, loss=loss, w=w, wloss=w * loss))

losses = np.array([s['loss'] for s in results])
print('N = %d scenarios' % len(results))
print('loss: min %.2f   median %.2f   max %.2f   share with a loss > 0: %.1f%%'
      % (losses.min(), np.median(losses), losses.max(), 100 * (losses > 0).mean()))


def show(tag, s):
    print('%s: X1 = %+.2f, X2 = %+.2f, sigma1/sigma1_0 = %.2f, sigma2/sigma2_0 = %.2f, d rho = %+.1f'
          % (tag, s['X1'], s['X2'], s['k1'], s['k2'], s['drho']))
    print('    loss = %.4f   weight = %.6f (= %s)   w x loss = %.4f'
          % (s['loss'], s['w'], Fraction(s['w']).limit_denominator(1000), s['wloss']))


def table(rows, key):
    print('  %6s %6s %6s %6s %6s %10s %8s %10s' % ('X1', 'X2', 'k1', 'k2', 'drho', 'loss', 'w', 'w*loss'))
    for s in sorted(rows, key=lambda s: -s[key])[:8]:
        print('  %+6.2f %+6.2f %6.2f %6.2f %+6.1f %10.2f %8.4f %10.2f'
              % (s['X1'], s['X2'], s['k1'], s['k2'], s['drho'], s['loss'], s['w'], s['wloss']))

N = 3888 scenarios
loss: min -8959.53   median -1382.49   max 1000.36   share with a loss > 0: 12.1%


### (b) Worst-case scenario risk measure

In [7]:
worst = max(results, key=lambda s: s['loss'])
show('(b) max loss', worst)
print('\neight largest losses')
table(results, 'loss')

(b) max loss: X1 = +0.20, X2 = +0.20, sigma1/sigma1_0 = 0.50, sigma2/sigma2_0 = 0.50, d rho = +0.5
    loss = 1000.3617   weight = 0.078125 (= 5/64)   w x loss = 78.1533

eight largest losses
      X1     X2     k1     k2   drho       loss        w     w*loss
   +0.20  +0.20   0.50   0.50   +0.5    1000.36   0.0781      78.15
   +0.20  +0.20   0.75   0.50   +0.5     983.14   0.1953     192.02
   +0.20  +0.20   0.75   0.75   +0.5     866.08   0.4883     422.89
   +0.10  +0.10   0.50   0.50   +0.5     845.82   0.1758     148.68
   +0.20  +0.20   0.50   0.50   +0.0     844.20   0.0312      26.38
   +0.10  +0.10   0.75   0.50   +0.5     831.73   0.4395     365.50
   +0.20  +0.20   0.50   0.75   +0.5     825.35   0.1953     161.20
   +0.20  +0.20   1.25   0.50   +0.5     787.04   0.1172      92.23


### (c) Weighted scenario risk measure

In [8]:
worst_w = max(results, key=lambda s: s['wloss'])
show('(c) max weighted loss', worst_w)
print('\neight largest weighted losses')
table(results, 'wloss')

w_max = max(s['w'] for s in results)
print('\nlargest weight on the grid = %.6f = %s, attained by %d scenarios'
      % (w_max, Fraction(w_max).limit_denominator(1000), sum(s['w'] == w_max for s in results)))
print('the (c) scenario has the largest weight: %s' % (worst_w['w'] == w_max))
print('its loss is %.1f%% of the (b) worst case' % (100 * worst_w['loss'] / worst['loss']))

(c) max weighted loss: X1 = +0.05, X2 = +0.05, sigma1/sigma1_0 = 0.75, sigma2/sigma2_0 = 0.75, d rho = +0.5
    loss = 672.6584   weight = 1.953125 (= 125/64)   w x loss = 1313.7859

eight largest weighted losses
      X1     X2     k1     k2   drho       loss        w     w*loss
   +0.05  +0.05   0.75   0.75   +0.5     672.66   1.9531    1313.79
   -0.05  -0.05   0.75   0.75   +0.5     550.52   1.9531    1075.23
   +0.10  +0.05   0.75   0.75   +0.5     633.31   1.4648     927.70
   +0.05  +0.10   0.75   0.75   +0.5     617.26   1.4648     904.19
   +0.10  +0.10   0.75   0.75   +0.5     735.89   1.0986     808.47
   +0.05  +0.05   1.75   0.75   +0.5     390.88   1.9531     763.43
   +0.05  +0.05   1.25   0.75   +0.5     613.70   1.1719     719.18
   -0.10  -0.05   0.75   0.75   +0.5     485.07   1.4648     710.55

largest weight on the grid = 1.953125 = 125/64, attained by 32 scenarios
the (c) scenario has the largest weight: True
its loss is 67.2% of the (b) worst case


### Where the losses come from

Split the (b) and (c) losses into the price move alone (parameters unchanged) and the parameter change at the new prices. The variance-form approximation of the price change $\Delta V \approx e^{r\tau}\tau\,\Delta\big(s_1^2\sigma_1^2 - 2\rho\sigma_1\sigma_2 s_1 s_2 + s_2^2\sigma_2^2\big)$ is reported next to it.

In [9]:
def decompose(s):
    S1, S2 = S1_0 * np.exp(s['X1']), S2_0 * np.exp(s['X2'])
    sig1, sig2, rho = s['k1'] * sigma1_0, s['k2'] * sigma2_0, rho_0 + s['drho']
    port = lambda a, b, c: V(h, S1, S2, a, b, c) - h1 * S1 - h2 * S2
    l_move = -(port(sigma1_0, sigma2_0, rho_0) - V_0)              # prices move, parameters stay
    l_param = -(port(sig1, sig2, rho) - port(sigma1_0, sigma2_0, rho_0))   # parameters change at the new prices
    tau = T - h
    var_old = sigma1_0**2 * S1**2 - 2 * rho_0 * sigma1_0 * sigma2_0 * S1 * S2 + sigma2_0**2 * S2**2
    var_new = sig1**2 * S1**2 - 2 * rho * sig1 * sig2 * S1 * S2 + sig2**2 * S2**2
    approx = -np.exp(r * tau) * tau * (var_new - var_old)
    print('  price move only: %8.2f   parameter change: %8.2f   total: %8.2f   (variance approximation of the parameter part: %8.2f)'
          % (l_move, l_param, l_move + l_param, approx))


print('(b) scenario'); decompose(worst)
print('(c) scenario'); decompose(worst_w)

# the price move alone, over the whole X grid at the t = 0 parameters: the hedge leaves gamma only
base = [s for s in results if s['k1'] == 1.25 and s['k2'] == 1.25 and s['drho'] == 0.0]
print('\nloss from the price move alone (parameters unchanged), X1 down the rows, X2 across')
print('        ' + ''.join('%8.2f' % x for x in X_grid))
for x1 in X_grid:
    S1 = S1_0 * np.exp(x1)
    row = []
    for x2 in X_grid:
        S2 = S2_0 * np.exp(x2)
        row.append(-(V(h, S1, S2, sigma1_0, sigma2_0, rho_0) - h1 * S1 - h2 * S2 - V_0))
    print('  %+.2f ' % x1 + ''.join('%8.2f' % v for v in row))

(b) scenario
  price move only:    47.59   parameter change:   952.77   total:  1000.36   (variance approximation of the parameter part:   941.52)
(c) scenario
  price move only:    66.31   parameter change:   606.35   total:   672.66   (variance approximation of the parameter part:   599.32)

loss from the price move alone (parameters unchanged), X1 down the rows, X2 across
           -0.20   -0.10   -0.05    0.05    0.10    0.20
  -0.20    11.40 -119.47 -372.70-1354.21-2129.66-4389.67
  -0.10  -228.62   42.57    5.96 -508.42-1032.19-2749.47
  -0.05  -513.84  -26.03   54.08 -208.63 -596.80-2021.68
  +0.05 -1500.44 -545.50 -213.72   66.31  -29.46 -823.78
  +0.10 -2242.43-1035.82 -568.44    3.99   65.76 -388.82
  +0.20 -4344.05-2594.70-1834.92 -631.93 -230.43   47.59
